# 01. LangChain 소개

**LangChain v1 프레임워크 개요**

## 학습 목표

LangChain 프레임워크의 구조와 핵심 컴포넌트를 이해합니다.

이 노트북을 마치면 다음을 알게 됩니다:

- LangChain v1 프레임워크의 3가지 레이어 구조
- ReAct 에이전트 패턴의 동작 방식
- 핵심 컴포넌트와 주요 API
- 개발 환경 설정 및 검증 방법

## 1.1 LangChain 프레임워크 개요

LangChain v1은 LLM 기반 에이전트를 구축하는 통합 프레임워크입니다. 3가지 레이어로 구성되며, 각 레이어는 서로 다른 수준의 추상화를 제공합니다.

### 3가지 레이어 구조

![LangChain 3계층 아키텍처](../assets/images/langchain_3layer.png)

| 레이어 | 역할 | 대상 사용자 |
|--------|------|------------|
| **LangChain** | 에이전트 생성의 핵심 API (`create_agent`, `tool`, `ChatOpenAI`) | 모든 개발자 |
| **LangGraph** | 복잡한 워크플로 구현 (상태 그래프, 체크포인터, 스트리밍) | 중급 이상 |
| **Deep Agents** | 사전 구축된 에이전트 (코딩, 리서치 등) | 빠른 프로토타이핑 |

### LangChain v1에서 변경된 주요 사항

| 항목 | 이전 (v0.x) | 현재 (v1) |
|------|-------------|----------|
| 에이전트 생성 | `create_react_agent()` | **`create_agent()`** |
| 에이전트 임포트 | `from langchain.agents import ...` (다양) | **`from langchain.agents import create_agent`** |
| 모델 초기화 | `ChatOpenAI(...)` 직접 사용 | `init_chat_model()` 또는 `ChatOpenAI(...)` |
| 메모리 | `ConversationBufferMemory` 등 | **`InMemorySaver`** (LangGraph 체크포인터) |
| 실행 엔진 | AgentExecutor | **LangGraph 그래프** (내부적으로) |

### 핵심 설계 철학

LangChain v1의 핵심은 **모든 에이전트가 LangGraph 그래프로 실행된다**는 점입니다. `create_agent()`로 생성된 에이전트는 내부적으로 LangGraph의 `StateGraph`로 구현되며, 이로 인해:

- **스트리밍**: `stream()` 메서드로 실시간 응답
- **상태 관리**: 체크포인터를 통한 대화 히스토리 유지
- **확장성**: 커스텀 노드와 엣지 추가 가능

## 1.2 ReAct 에이전트 패턴

ReAct (Reasoning + Acting) 패턴은 LangChain v1 에이전트의 기본 동작 방식입니다. 에이전트는 다음 루프를 반복합니다:

![ReAct 루프 — 추론→행동→관찰 반복](../assets/images/react_loop.png)

### 핵심 특징

- **자율적 판단**: 에이전트가 도구 사용 여부를 스스로 결정합니다
- **다단계 추론**: 복잡한 작업을 여러 단계로 분해하여 처리합니다
- **관찰 기반 학습**: 도구 결과를 관찰하고 다음 행동을 결정합니다

## 1.3 주요 컴포넌트 개요

LangChain v1의 핵심 컴포넌트를 표로 정리합니다:

| 컴포넌트 | 설명 | 주요 API |
|----------|------|----------|
| **Model** | LLM 또는 Chat 모델. 에이전트의 "두뇌" 역할 | `ChatOpenAI`, `init_chat_model()` |
| **Tools** | 에이전트가 사용할 수 있는 함수. 검색, 계산, API 호출 등 | `@tool` 데코레이터, `TavilySearch` |
| **Agent** | 모델과 도구를 결합한 실행 단위. 내부적으로 LangGraph 그래프 | `create_agent()` |
| **Memory** | 대화 히스토리를 저장하고 관리하는 체크포인터 | `InMemorySaver`, `SqliteSaver` |
| **Middleware** | 요청/응답 처리 파이프라인에 로직 삽입 | `prompt`, `before_tool`, `after_model` |
| **State** | 에이전트 실행 중 관리되는 상태 (메시지, 컨텍스트 등) | `AgentState`, `messages` |
| **Streaming** | 실시간 응답 스트리밍 지원 | `stream()`, `stream_mode="updates"` |

## 1.4 환경 설정 및 설치 확인

LangChain v1 개발에 필요한 패키지와 API 키를 확인합니다.

In [1]:
# 환경 설정
import subprocess
import importlib

packages = {
    "langchain": "langchain",
    "langchain_openai": "langchain-openai",
    "langchain_community": "langchain-community",
    "langgraph": "langgraph",
}

print("=" * 50)
print("LangChain v1 환경 확인")
print("=" * 50)

for module_name, package_name in packages.items():
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "installed")
        print(f"\u2713 {package_name}: {version}")
    except ImportError:
        print(f"\u2717 {package_name}: 미설치 \u2192 pip install {package_name}")

LangChain v1 환경 확인
✓ langchain: 1.2.10


✓ langchain-openai: installed
✓ langchain-community: 0.4.1
✓ langgraph: installed


In [2]:
# API 키 확인
from dotenv import load_dotenv
import os

load_dotenv(override=True)

required_keys = ["OPENAI_API_KEY"]
optional_keys = ["TAVILY_API_KEY", "LANGSMITH_API_KEY"]

print("필수 API 키:")
for key in required_keys:
    status = "\u2713 설정됨" if os.environ.get(key) else "\u2717 미설정"
    print(f"  {key}: {status}")

print("\n선택 API 키:")
for key in optional_keys:
    status = "\u2713 설정됨" if os.environ.get(key) else "- 미설정 (선택)"
    print(f"  {key}: {status}")

필수 API 키:
  OPENAI_API_KEY: ✓ 설정됨

선택 API 키:
  TAVILY_API_KEY: ✓ 설정됨
  LANGSMITH_API_KEY: - 미설정 (선택)


In [3]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


In [4]:
# LangChain v1 핵심 import 확인
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

print("\u2713 모든 핵심 모듈 임포트 성공")
print("  - create_agent: LangChain v1 에이전트 생성")
print("  - tool: 도구 데코레이터")
print("  - ChatOpenAI: OpenAI 호환 모델")
print("  - InMemorySaver: 메모리 체크포인터")

✓ 모든 핵심 모듈 임포트 성공
  - create_agent: LangChain v1 에이전트 생성
  - tool: 도구 데코레이터
  - ChatOpenAI: OpenAI 호환 모델
  - InMemorySaver: 메모리 체크포인터


## 1.5 다음 단계

LangChain v1 프레임워크의 구조와 핵심 컴포넌트를 살펴봤습니다.

다음 노트북 (`02_quickstart.ipynb`)에서는 실제로 에이전트를 생성하고 실행합니다:

- `@tool` 데코레이터로 커스텀 도구 정의
- `create_agent()`로 에이전트 생성
- `invoke()`와 `stream()`으로 에이전트 실행
- `InMemorySaver`로 멀티턴 대화 구현

## 요약

| 항목 | 설명 |
|------|------|
| LangChain v1 | 에이전트 중심 프레임워크, `create_agent()` API 기반 |
| 3계층 구조 | 모델 → 도구 → 미들웨어 |
| ReAct 패턴 | 추론(Reasoning) → 행동(Action) → 관찰(Observation) 반복 |
| 핵심 API | `create_agent()`, `@tool`, `invoke()`, `stream()` |

### 다음 단계
→ **[02_quickstart.ipynb](./02_quickstart.ipynb)**: LangChain 첫 번째 에이전트를 만듭니다.
